# Movie Recommendation Engine

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### The Core Theory Behind the Recommendation Engine

**1. Content-Based Filtering vs. Collaborative Filtering**
This engine is a **Content-Based Recommender**. It looks *inward* at the items themselves to figure out what a movie is about, and finds other movies about the exact same things. It ignores other users (which would be *Collaborative Filtering*).

**2. Feature Selection (Finding the "DNA")**
We select features that make up the movie's DNA: **Genres, Keywords, Cast, and Director**. We drop numerical data like Budget or Runtime because they don't define the *plot or style* of the movie.

**3. The "Bag of Words" Concept**
We throw all these features into one giant string of text. We lose sentence structure and just have a "bag" containing words like: *Christopher, Nolan, Sci-Fi, Space*.

**4. Vector Space Model (Turning Words into Math)**
Using **Count Vectorization**, we take our entire vocabulary and map it to a high-dimensional coordinate system. Every unique word is an axis, and every movie becomes a single point (a vector) based on its word frequencies.

**5. Cosine Similarity (Why Angles Matter)**
To measure how close two movies are, we don't use straight-line distance (**Euclidean Distance**) because a movie with 50 keywords might be physically "far away" from a similar movie with only 5 keywords. Instead, we measure the angle between their vectors (**Cosine Similarity**). If they point in the exact same direction, they share the same proportional mix of features, regardless of their sheer size!

In [3]:
def get_title_from_index(index, df):
    return df[df.index == index]["title"].values[0]

def get_index_from_title(title, df):
    # Try case-insensitive matching if exact match not found
    matches = df[df.title.str.lower() == title.lower()]
    if len(matches) > 0:
        return matches.index[0]
    return None

def combine_features(row):
    try:
        return row['keywords'] + " " + row['cast'] + " " + row['genres'] + " " + row['director']
    except:
        print("Error:", row)
        return ""


In [5]:
# Step 1: Read CSV File
try:
    df = pd.read_csv("movie_dataset.csv")
except FileNotFoundError:
    print("Error: movie_dataset.csv not found. Please ensure the dataset is present.")

# Step 2: Select Features
features = ['keywords', 'cast', 'genres', 'director']

# Step 3: Create a column in DF which combines all selected features
for feature in features:
    df[feature] = df[feature].fillna('')

df["combined_features"] = df.apply(combine_features, axis=1)  #combined feature function
display(df[["combined_features"]].head(5))

# Step 4: Create count matrix from this new combined column
cv = CountVectorizer()
count_matrix = cv.fit_transform(df["combined_features"])

# Step 5: Compute the Cosine Similarity based on the count_matrix
cosine_sim = cosine_similarity(count_matrix) 

,combined_features
0,culture clash future space war space colony so...
1,ocean drug abuse exotic island east india trad...
2,spy based on novel secret agent sequel mi6 Dan...
3,dc comics crime fighter terrorist secret ident...
4,based on novel mars medallion space travel pri...


In [9]:
movie_user_likes = "Avatar"

# Step 6: Get index of this movie from its title
movie_index = get_index_from_title(movie_user_likes, df)
if movie_index is None:
    print(f"Sorry, movie '{movie_user_likes}' not found in the dataset.")
else:
    similar_movies = list(enumerate(cosine_sim[movie_index]))

    # Step 7: Get a list of similar movies in descending order of similarity score
    sorted_similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)[1:]

    # Step 8: Print titles of first 10 movies
    print(f"\nTop 10 Recommendations for '{get_title_from_index(movie_index, df)}':")
    print("-" * 50)
    for i, element in enumerate(sorted_similar_movies[:10]):
        print(f"{i + 1}. {get_title_from_index(element[0], df)}")
    print("-" * 50)



Top 10 Recommendations for 'Avatar':
--------------------------------------------------
1. Guardians of the Galaxy
2. Aliens
3. Star Wars: Clone Wars: Volume 1
4. Star Trek Into Darkness
5. Star Trek Beyond
6. Alien
7. Lockout
8. Jason X
9. The Helix... Loaded
10. Moonraker
--------------------------------------------------
